# Capítulo 8: Regressão Linear

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 3 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [8.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/01-regressao-linear-simples.html) | Regressão Linear Simples |
| [8.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/02-avaliando-o-ajuste.html) | Avaliando o Ajuste: R² e Erro |
| [8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-regressao-multipla.html) | Regressão Múltipla |
| [8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-preditores-qualitativos.html) | Preditores Qualitativos |
| [8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-interacao-e-termos-nao-lineares.html) | Interação e Termos Não Lineares |
| [8.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-outliers-alavancagem-e-colinearidade.html) | Outliers, Alavancagem e Colinearidade |
| [8.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/07-regressao-linear-contra-k-vizinhos.html) | Regressão Linear contra k-Vizinhos |

## Regressão Linear Simples

> **📌 Nota**
>
> Esta seção corresponde às seções 3.1 e 3.1.1 de James et al. (2023).

`Advertising` traz o quanto duzentos mercados investiram em três mídias de propaganda — televisão, rádio e jornal — e quantas unidades do produto cada um vendeu. A pergunta mais simples que esse dado permite fazer é também a primeira: o investimento em TV, sozinho, ajuda a prever vendas? Regressão linear simples responde ajustando uma reta a exatamente dois números por mercado, `tv` e `vendas`, deixando `radio` e `jornal` de fora por enquanto.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

plt.style.use("estilo-figuras.mplstyle")

### Uma reta para tv e vendas

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
propaganda.shape, propaganda.columns.tolist()

Duzentos mercados, quatro colunas — `tv`, `radio`, `jornal` e `vendas`, a primeira em milhares de dólares, a última em milhares de unidades. Regressão linear simples escreve a relação entre as duas que interessam aqui como

$$
\text{vendas} \approx \beta_0 + \beta_1 \cdot \text{tv}.
$$

$\beta_0$ e $\beta_1$ são duas constantes, desconhecidas até se estimar: $\beta_0$ é o **intercepto** — o valor esperado de vendas quando o investimento em TV é zero —, e $\beta_1$ é a **inclinação** — o quanto vendas muda, em média, para cada unidade a mais de tv. O símbolo "≈" marca que a relação é uma aproximação: nada garante que dois mercados com o mesmo investimento em TV vendam exatamente o mesmo, e o quanto cada um foge da reta é o que o resto desta seção mede.

### O resíduo, e a soma que ele eleva ao quadrado

Uma vez que se tem estimativas $\hat\beta_0$ e $\hat\beta_1$, a reta prevê

$$
\hat y_i = \hat\beta_0 + \hat\beta_1 x_i,
$$

e o **resíduo** daquele mercado é a distância entre o que ele de fato vendeu e o que a reta previu para o mesmo investimento em TV:

$$
e_i = y_i - \hat y_i.
$$

A **soma dos quadrados dos resíduos** (RSS) soma esse erro, ao quadrado, sobre os duzentos mercados:

$$
\text{RSS} = e_1^2 + e_2^2 + \cdots + e_n^2 = \sum_{i=1}^{n} \left(y_i - \hat\beta_0 - \hat\beta_1 x_i\right)^2.
$$

Elevar ao quadrado, em vez de somar o valor absoluto de cada resíduo, pune um erro grande desproporcionalmente mais do que vários erros pequenos, e deixa RSS como uma soma de parábolas em $\beta_0$ e $\beta_1$ — uma superfície com um único fundo, que se acha por fórmula fechada em vez de busca. Ajustar a reta é escolher, entre todos os pares $(\beta_0, \beta_1)$ possíveis, o único que minimiza essa soma: é a esse critério que se dá o nome de **mínimos quadrados**.

> **🔷 Conceito**
>
> O **resíduo** $e_i = y_i - \hat y_i$ mede a distância entre um valor observado e o previsto pela reta. A **soma dos quadrados dos resíduos** (RSS) soma $e_i^2$ sobre todas as observações. **Mínimos quadrados** é o critério que escolhe $\hat\beta_0$ e $\hat\beta_1$ minimizando RSS — nenhum outro par de coeficientes produz uma reta com RSS menor.

### A conta à mão: duas médias bastam

Minimizar RSS por cálculo — derivando em relação a $\beta_0$ e a $\beta_1$ e igualando as duas derivadas a zero — leva a uma fórmula fechada que depende só das médias de `tv` e `vendas`, e dos desvios de cada ponto em relação a elas:

$$
\hat\beta_1 = \frac{\displaystyle\sum_{i=1}^{n} (x_i - \bar x)(y_i - \bar y)}{\displaystyle\sum_{i=1}^{n} (x_i - \bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x.
$$

Não precisa de nenhuma biblioteca de otimização — dá para calcular direto com `numpy`:

In [ ]:
tv = propaganda["tv"]
vendas = propaganda["vendas"]

tv_media = tv.mean()
vendas_media = vendas.mean()
beta1_mao = ((tv - tv_media) * (vendas - vendas_media)).sum() / ((tv - tv_media) ** 2).sum()
beta0_mao = vendas_media - beta1_mao * tv_media

round(tv_media, 2), round(vendas_media, 2), round(beta1_mao, 4), round(beta0_mao, 4), round(beta1_mao * 1000, 1)

O investimento médio em TV é 147,04 (mil dólares); a venda média, 14,02 (mil unidades). A partir só dessas duas médias e dos desvios em relação a elas, $\hat\beta_1$ sai 0,0475 e $\hat\beta_0$, 7,0326. Como `tv` e `vendas` vêm as duas em milhares, $\hat\beta_1 \times 1.000$ traduz a inclinação para a escala do dinheiro gasto: 47,5 — cada mil dólares a mais investidos em TV está associado, em média, a 47,5 unidades a mais vendidas.

### A mesma conta, pronta: `LinearRegression`

O `scikit-learn` resolve a mesma minimização sem passar pelas médias explicitamente. `LinearRegression().fit(X, y)` recebe o preditor e a resposta e devolve um objeto já ajustado, com a inclinação em `.coef_` e o intercepto em `.intercept_` — os dois como array e escalar do `numpy`, por isso o `float(...)` ao redor de cada um daqui em diante. `X` entra como o `DataFrame` que já veio do `pandas`, mesmo sendo de uma coluna só, sem nenhum `.to_numpy()`: o estimador aceita e devolve, em `.feature_names_in_`, o nome de coluna que recebeu.

In [ ]:
X = propaganda[["tv"]]
y = propaganda["vendas"]

modelo = LinearRegression().fit(X, y)
beta1_sklearn = float(modelo.coef_[0])
beta0_sklearn = float(modelo.intercept_)

(
    modelo.feature_names_in_,
    (round(beta0_mao, 4), round(beta1_mao, 4)),
    (round(beta0_sklearn, 4), round(beta1_sklearn, 4)),
    bool(np.allclose([beta0_mao, beta1_mao], [beta0_sklearn, beta1_sklearn])),
)

`feature_names_in_` guarda só `tv`, o único nome que o `DataFrame` de uma coluna carregava. Os dois pares de coeficiente — o calculado à mão e o que saiu do `.fit()` — são (7,0326; 0,0475) nos dois casos, e `np.allclose` confirma: `True`. É a mesma fórmula fechada por trás das duas contas; a segunda só evita escrever as médias à mão.

### A reta, e o resíduo que ela deixa

In [ ]:
# Figura: Duzentos mercados: vendas contra o investimento em TV, com a reta de mínimos quadrados por cima. Cada segmento liga um mercado observado à previsão da reta para o mesmo investimento — o resíduo daquele mercado, o e_i que RSS eleva ao quadrado.
yhat = modelo.predict(X)
grade_tv = np.linspace(tv.min(), tv.max(), 200)
reta_grade = modelo.predict(pd.DataFrame({"tv": grade_tv}))

fig, ax = plt.subplots()
ax.vlines(tv, np.minimum(vendas, yhat), np.maximum(vendas, yhat), color="C1", linewidth=1)
ax.plot(grade_tv, reta_grade, color="C0", linewidth=2, label="reta ajustada")
ax.scatter(tv, vendas, color="C2", s=18, zorder=3, label="observado")
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("vendas (milhares de unidades)")
ax.legend()
plt.tight_layout()
plt.show()

A reta captura a tendência — vender mais conforme se investe mais em TV —, mas nenhum mercado senta exatamente sobre ela: sempre sobra um segmento. Somar o quadrado dos duzentos segmentos desta figura dá exatamente o RSS que a reta minimiza:

In [ ]:
rss_min = float(((vendas - yhat) ** 2).sum())
round(rss_min, 2)

2.102,53 — nenhuma outra reta, entre todos os pares $(\beta_0, \beta_1)$ possíveis, soma menos que isso.

### Um vale com um fundo só

RSS, como soma de quadrados de uma função linear de $\beta_0$ e $\beta_1$, é uma superfície convexa nesses dois parâmetros: um paraboloide elíptico, sem platôs nem mínimos locais além de um único ponto. Variando $\beta_0$ e $\beta_1$ numa grade ao redor de $(\hat\beta_0, \hat\beta_1)$ e calculando RSS em cada combinação, essa forma aparece em curvas de nível — cada uma liga os pares que produzem o mesmo RSS.

In [ ]:
# Figura: Curvas de nível de RSS sobre (β0, β1), na regressão de vendas sobre tv em Advertising. O ponto marcado é (β̂0, β̂1); cada curva liga pares com o mesmo RSS, e as seis se fecham ao redor desse único ponto — não há outro vale na janela.
grade_beta0 = np.linspace(3.5, 10.5, 300)
grade_beta1 = np.linspace(0.02, 0.075, 300)
malha_beta0, malha_beta1 = np.meshgrid(grade_beta0, grade_beta1)

tv_np = tv.to_numpy()
vendas_np = vendas.to_numpy()
residuo_grade = vendas_np - malha_beta0[:, :, None] - malha_beta1[:, :, None] * tv_np
rss_grade = (residuo_grade ** 2).sum(axis=2)

niveis_rss = np.array([2150.0, 2200.0, 2300.0, 2400.0, 2500.0, 2600.0])

fig, ax = plt.subplots()
contornos = ax.contour(malha_beta0, malha_beta1, rss_grade, levels=niveis_rss, cmap="Blues")
ax.scatter([beta0_sklearn], [beta1_sklearn], color="C1", s=40, zorder=3)
ax.annotate(
    r"$(\hat\beta_0,\ \hat\beta_1)$",
    xy=(beta0_sklearn, beta1_sklearn),
    xytext=(10, -14),
    textcoords="offset points",
    fontsize=9,
)
ax.set_xlabel(r"$\beta_0$")
ax.set_ylabel(r"$\beta_1$")
barra = fig.colorbar(contornos, ax=ax, shrink=0.9, pad=0.02)
barra.set_label("RSS")
plt.tight_layout()
plt.show()

A legenda promete curvas fechadas, então a afirmação se confere contando, não olhando: o próprio objeto que o `contour` devolve guarda, para cada nível, os segmentos de linha que desenhou, e um segmento fecha quando termina no mesmo ponto em que começou.

In [ ]:
segmentos_totais = 0
segmentos_fechados = 0
for segmentos_do_nivel in contornos.allsegs:
    for segmento in segmentos_do_nivel:
        if len(segmento) == 0:
            continue
        segmentos_totais += 1
        if np.allclose(segmento[0], segmento[-1]):
            segmentos_fechados += 1

segmentos_totais, segmentos_fechados, segmentos_fechados == segmentos_totais

Seis níveis, seis segmentos desenhados, e os seis fecham: `segmentos_totais` e `segmentos_fechados` saem iguais, 6 e 6. Nenhuma curva sai cortada pela borda da janela — o que confirma, em vez de supor, que RSS tem um único vale nesta vizinhança de $(\hat\beta_0, \hat\beta_1)$.

In [ ]:
minimo_real_menor_que_grade = bool(rss_min < rss_grade.min())
round(rss_min, 2), round(float(rss_grade.min()), 2), minimo_real_menor_que_grade

O mínimo verdadeiro, 2.102,53, fica abaixo até do menor valor que a própria grade alcança, 2.102,56 — `minimo_real_menor_que_grade` é `True`. Nenhum dos 300×300 pontos testados coincide exatamente com $(\hat\beta_0, \hat\beta_1)$, só passa perto: é a fórmula fechada, não a grade, que encontra o fundo do vale de verdade. RSS diz qual par de coeficientes é o melhor entre os que este dado observou; não diz se essa reta presta para prever vendas em geral, nem quanto de vendas ela de fato explica.

## Avaliando o Ajuste: R² e Erro

> **📌 Nota**
>
> Esta seção corresponde à seção 3.1.3 de James et al. (2023).

Ajustar uma reta por mínimos quadrados sempre dá um $\hat\beta_0$ e um $\hat\beta_1$ — a seção anterior encontrou o par que minimiza RSS para `vendas ~ tv` e não sobrou nenhum outro candidato menor. A pergunta muda agora: o quanto essa reta *serve*. Duas medidas respondem isso, cada uma de um jeito, e as duas partem do mesmo RSS.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use("estilo-figuras.mplstyle")

propaganda = pd.read_csv("dados/Advertising.csv")
X = propaganda[["tv"]]
y = propaganda["vendas"]
modelo = LinearRegression().fit(X, y)
yhat = modelo.predict(X)

### O erro típico de previsão: RSE

O erro-padrão residual (RSE) resume o RSS como um único número, na mesma unidade da resposta. É a raiz do RSS dividido não por $n$, e sim por $n-2$ — a reta já consumiu dois parâmetros, $\hat\beta_0$ e $\hat\beta_1$, antes de sobrar erro para medir:

$$
\text{RSE} = \sqrt{\frac{\text{RSS}}{n-2}}
$$

In [ ]:
n = len(y)
n_menos_2 = n - 2
rss = float(((y - yhat) ** 2).sum())
rse_mao = float(np.sqrt(rss / n_menos_2))
rse_mil_unidades = round(rse_mao * 1000)

mse_sklearn = mean_squared_error(y, yhat)
rse_via_mse = float(np.sqrt(mse_sklearn * n / n_menos_2))

vendas_media = float(y.mean())
percentual_erro = rse_mao / vendas_media * 100

(
    n,
    n_menos_2,
    round(rss, 2),
    round(rse_mao, 4),
    round(rse_via_mse, 4),
    bool(np.isclose(rse_mao, rse_via_mse)),
    rse_mil_unidades,
    round(vendas_media, 4),
    round(percentual_erro, 2),
)

Duzentos mercados, `n_menos_2 = 198` depois de descontar os dois parâmetros que a reta gastou para se ajustar: RSS, 2.102,53, dividido por 198 e com a raiz, dá 3,2587. `mean_squared_error` calcula o mesmo RSS já dividido por $n$ — não por $n-2$ —, então recuperar o RSE a partir dele exige desfazer essa divisão antes de tirar a raiz; feito isso, os dois caminhos batem: 3,2587 nos dois casos, `True`.

Vendas está em milhares de unidades, então 3,2587 quer dizer que a venda observada de um mercado típico se afasta da reta em cerca de 3.259 unidades — para cima ou para baixo, em média. Contra a venda média de 14,0225 mil unidades, esses 3,2587 mil unidades representam 23,24% dela: é esse o tamanho do erro típico de previsão, relativo à própria escala do que se está prevendo.

### R²: a proporção de variância explicada

RSE vem na unidade de `vendas`, e 3,2587 só diz alguma coisa a quem sabe a escala de vendas — não dá para comparar direto com o RSE de um modelo cuja resposta é medida em outra unidade. R² resolve isso descartando a unidade: é a fração do TSS — a variância total da resposta, antes de qualquer reta — que a regressão explica.

$$
\text{TSS} = \sum_{i=1}^n (y_i - \bar y)^2, \qquad R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}
$$

In [ ]:
tss = float(((y - vendas_media) ** 2).sum())
explicada = tss - rss

r2_mao = 1 - rss / tss
r2_sklearn = float(r2_score(y, yhat))
r2_via_score = float(modelo.score(X, y))

(
    round(tss, 2),
    round(explicada, 2),
    round(r2_mao, 4),
    round(r2_sklearn, 4),
    round(r2_via_score, 4),
    bool(np.isclose(r2_mao, r2_sklearn) and np.isclose(r2_mao, r2_via_score)),
    round(r2_mao * 100, 2),
)

TSS, a variância total de vendas antes de qualquer reta, sai 5.417,15; descontado o RSS de 2.102,53, sobram 3.314,62 de variância que a reta explica. A conta à mão, `r2_score` e `.score()` — os três caminhos para o mesmo número — concordam em 0,6119: `True`. Uma reta que usa só o investimento em TV explica 61,19% da variância de vendas, sem que esse número carregue unidade nenhuma — é isso que o torna comparável entre modelos onde o RSE não é.

> **🔷 Conceito**
>
> O **erro-padrão residual** (RSE) é a raiz de $\text{RSS}/(n-2)$: o tamanho típico do resíduo, na unidade da resposta. O **R²** é $1 - \text{RSS}/\text{TSS}$: a fração da variância da resposta que o modelo explica, sempre entre 0 e 1 e sem unidade — o que o torna comparável de um modelo para outro, mesmo quando o RSE não é.

R² alto não garante um modelo bom, nem R² baixo condena um modelo ruim: os dois dependem de quanto do problema é o $\mathrm{Var}(\epsilon)$ que o capítulo anterior chamou de piso irredutível — nenhuma reta, nem a melhor possível, explica a parte da variância que é ruído puro.

### O que a reta previu contra o que cada mercado vendeu

In [ ]:
soma_confere = bool(np.isclose(rss + explicada, tss))
explicada_maior_que_rss = bool(explicada > rss)
soma_confere, explicada_maior_que_rss

RSS mais a parte explicada soma de volta o TSS — `soma_confere` é `True` — e a parte explicada é maior que a não explicada, `explicada_maior_que_rss` também `True`, o mesmo fato que R² > 0,5 já dizia.

In [ ]:
# Figura: À esquerda, previsto contra observado para os duzentos mercados de Advertising, com a diagonal de previsão perfeita: a distância vertical de cada ponto até ela é o resíduo daquele mercado. À direita, o TSS decomposto em RSS (não explicada) e a parte que a reta explica, com R² anotado como a fração de cima.
fig, (ax_diag, ax_barra) = plt.subplots(1, 2, figsize=(9, 4.2))

limite = (min(y.min(), yhat.min()) - 1, max(y.max(), yhat.max()) + 1)
ax_diag.plot(limite, limite, color="C1", linewidth=1.5, label="previsão perfeita")
ax_diag.scatter(y, yhat, color="C0", s=16, alpha=0.7, label="mercado")
ax_diag.set_xlim(limite)
ax_diag.set_ylim(limite)
ax_diag.set_aspect("equal")
ax_diag.set_xlabel("vendas observadas (mil unidades)")
ax_diag.set_ylabel("vendas previstas (mil unidades)")
ax_diag.legend()

ax_barra.bar(0, rss, color="C1", label="RSS (não explicada)")
ax_barra.bar(0, explicada, bottom=rss, color="C0", label="explicada (TSS − RSS)")
ax_barra.annotate(
    f"R² = {r2_mao:.3f}",
    xy=(0, rss + explicada / 2),
    xytext=(0.55, rss + explicada / 2),
    va="center",
    fontsize=9,
    arrowprops={"arrowstyle": "-", "linewidth": 0.8},
)
ax_barra.set_xlim(-0.6, 1.6)
ax_barra.set_xticks([0])
ax_barra.set_xticklabels(["TSS"])
ax_barra.set_ylabel("soma de quadrados")
ax_barra.legend(loc="upper right")

plt.tight_layout()
plt.show()

À esquerda, cada ponto é um mercado: quanto mais perto da diagonal, menor o resíduo que a seção anterior somou ao quadrado para formar o RSS. À direita, a mesma barra que soma TSS — verificado acima — dividida nas duas parcelas que R² compara: o pedaço laranja é o RSS que a reta deixou sem explicar, o azul é a fatia que R² = 0,612 mede como proporção do todo.

## Regressão Múltipla

> **📌 Nota**
>
> Esta seção corresponde à seção 3.2 de James et al. (2023).

`Advertising` traz três mídias, e a seção 8.1 usou só uma. A pergunta mais direta é repetir a mesma reta duas vezes mais — uma para `radio`, outra para `jornal` — e ler os três coeficientes lado a lado. Esta seção começa por essa pergunta, e termina desfazendo a resposta que ela parece dar.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.style.use("estilo-figuras.mplstyle")

propaganda = pd.read_csv("dados/Advertising.csv")
y = propaganda["vendas"]

### Jornal, sozinho, parece importar

In [ ]:
X_jornal = propaganda[["jornal"]]
modelo_jornal = LinearRegression().fit(X_jornal, y)
beta1_jornal_sozinho = float(modelo_jornal.coef_[0])

round(beta1_jornal_sozinho, 4), round(beta1_jornal_sozinho * 1000, 1)

Ajustada sozinha contra `jornal`, a reta de mínimos quadrados dá um coeficiente de 0,0547: cada mil dólares a mais gastos em jornal está associado, em média, a 54,7 unidades a mais vendidas. É um coeficiente positivo, do mesmo tipo que a seção 8.1 encontrou para `tv` — nada nesta regressão isolada distingue jornal das outras duas mídias.

Só que ajustar `jornal` sozinho ignora `tv` e `radio` por completo. Se as três mídias variam juntas de mercado para mercado, o coeficiente de uma pode estar levando crédito por vendas que outra produziu — e a única forma de saber é colocar as três na mesma equação.

### As três mídias na mesma equação

O modelo múltiplo estende a mesma ideia de mínimos quadrados a mais de um preditor: em vez de uma reta, um hiperplano que minimiza a soma dos quadrados dos resíduos sobre `tv`, `radio` e `jornal` ao mesmo tempo.

In [ ]:
X_multiplo = propaganda[["tv", "radio", "jornal"]]
modelo_multiplo = LinearRegression().fit(X_multiplo, y)

coeficientes = pd.Series(modelo_multiplo.coef_, index=modelo_multiplo.feature_names_in_)
coeficientes_mil_dolares = (coeficientes * 1000).round(1)

coeficientes.round(4), coeficientes_mil_dolares

`feature_names_in_` guarda os três nomes de coluna que o `DataFrame` carregava, na mesma ordem dos coeficientes — por isso dá para juntar os dois numa única `Series` em vez de decorar qual número é qual. `tv` sai com 0,0458 (45,8 por mil dólares) e `radio` com 0,1885 (188,5 por mil dólares), os dois positivos. `jornal` sai com -0,0010 (-1,0 por mil dólares): praticamente zero, e de sinal oposto ao 0,0547 que a regressão sozinha tinha encontrado para ele.

> **🔷 Conceito**
>
> Num modelo múltiplo, cada coeficiente $\hat\beta_j$ se lê como o efeito médio sobre a resposta de aumentar o preditor $X_j$ em uma unidade, **mantendo todos os outros preditores fixos**. É essa cláusula — mantendo os demais fixos — que separa a leitura de um coeficiente múltiplo da de uma regressão simples, e é ela que muda o que se pode dizer sobre `jornal`.

Lido dessa forma: gastar mais mil dólares em `tv`, mantendo `radio` e `jornal` fixos, está associado a 45,8 unidades a mais de venda; mais mil dólares em `radio`, mantendo `tv` e `jornal` fixos, a 188,5 unidades a mais. Mais mil dólares em `jornal`, mantendo `tv` e `radio` fixos, está associado a uma variação de -1,0 unidade — perto o bastante de zero para não sobrar efeito de jornal depois que as outras duas mídias já estão na conta.

### Por que jornal some: quem anda com quem

A explicação não está em mais uma regressão — está em como as três mídias se relacionam entre si, antes de qualquer venda entrar na conta.

In [ ]:
correlacoes = propaganda[["tv", "radio", "jornal", "vendas"]].corr()
correlacoes.round(4)

In [ ]:
corr_jornal_radio = float(correlacoes.loc["jornal", "radio"])
corr_jornal_tv = float(correlacoes.loc["jornal", "tv"])
jornal_mais_correlacionado_com_radio = bool(corr_jornal_radio > corr_jornal_tv)

round(corr_jornal_radio, 4), round(corr_jornal_tv, 4), jornal_mais_correlacionado_com_radio

`jornal` correlaciona com `radio` a 0,3541 e com `tv` a só 0,0566 — `jornal_mais_correlacionado_com_radio` confirma que a primeira supera a segunda. Mercados que gastam mais em rádio tendem a gastar mais em jornal também: os dois investimentos sobem e descem juntos, sem que isso implique nada causal entre eles.

É essa correlação que explica a reviravolta da seção anterior. Se é o rádio — não o jornal — que de fato move vendas, então nos mercados onde se gasta mais em rádio as vendas tendem a ser maiores, e a tabela de correlação mostra que esses são os mesmos mercados que gastam mais em jornal. Uma regressão que olha só para `jornal`, sem `radio` por perto, não tem como separar as duas coisas: ela atribui a jornal parte do crédito que é do rádio. Colocar as duas mídias na mesma equação é o que permite distinguir — e é aí que o coeficiente de jornal cai a praticamente zero.

### R² e RSE: o quanto o modelo múltiplo melhora

In [ ]:
X_tv = propaganda[["tv"]]
modelo_tv = LinearRegression().fit(X_tv, y)
yhat_tv = modelo_tv.predict(X_tv)

n = len(y)
rss_tv = float(((y - yhat_tv) ** 2).sum())
rse_tv = float(np.sqrt(rss_tv / (n - 2)))
r2_tv = float(r2_score(y, yhat_tv))

round(rse_tv, 4), round(r2_tv, 4)

In [ ]:
yhat_multiplo = modelo_multiplo.predict(X_multiplo)

p = X_multiplo.shape[1]
rss_multiplo = float(((y - yhat_multiplo) ** 2).sum())
rse_multiplo = float(np.sqrt(rss_multiplo / (n - p - 1)))
r2_multiplo = float(r2_score(y, yhat_multiplo))

round(rse_multiplo, 4), round(r2_multiplo, 4)

In [ ]:
comparacao = pd.DataFrame(
    {"R²": [r2_tv, r2_multiplo], "RSE": [rse_tv, rse_multiplo]},
    index=["tv sozinho", "tv + radio + jornal"],
).round(4)
comparacao

O modelo de `tv` sozinho explica 0,6119 da variância de vendas, com erro típico de 3,2587 mil unidades. Somar `radio` e `jornal` sobe o R² para 0,8972 e derruba o RSE para 1,6855: as três mídias juntas explicam mais da variância de vendas do que `tv` sozinho, e erram menos a cada previsão. A régua não muda — R² e RSE continuam sendo as mesmas duas medidas da seção anterior —, só o modelo que elas avaliam.

### A superfície ajustada com tv e radio

Como jornal quase não muda a previsão, a superfície que os coeficientes de `tv` e `radio` desenham já carrega quase toda a informação do modelo múltiplo — e, com só dois preditores, dá para desenhar essa superfície inteira.

In [ ]:
X_tv_radio = propaganda[["tv", "radio"]]
modelo_tv_radio = LinearRegression().fit(X_tv_radio, y)
yhat_tv_radio = modelo_tv_radio.predict(X_tv_radio)

r2_tv_radio = float(r2_score(y, yhat_tv_radio))
round(r2_tv_radio, 4)

In [ ]:
# Figura: Vendas previstas para toda combinação de investimento em tv e radio, pelo modelo ajustado com as duas mídias. Os pontos são os duzentos mercados observados.
grade_tv = np.linspace(propaganda["tv"].min(), propaganda["tv"].max(), 60)
grade_radio = np.linspace(propaganda["radio"].min(), propaganda["radio"].max(), 60)
malha_tv, malha_radio = np.meshgrid(grade_tv, grade_radio)
grade = pd.DataFrame({"tv": malha_tv.ravel(), "radio": malha_radio.ravel()})
superficie = modelo_tv_radio.predict(grade).reshape(malha_tv.shape)

fig, ax = plt.subplots()
mapa = ax.contourf(malha_tv, malha_radio, superficie, levels=14, cmap="Blues")
ax.scatter(propaganda["tv"], propaganda["radio"], color="C1", s=16, edgecolor="white", linewidth=0.5)
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("radio (milhares de dólares)")
barra = fig.colorbar(mapa, ax=ax, shrink=0.9, pad=0.02)
barra.set_label("vendas prevista (mil unidades)")
plt.tight_layout()
plt.show()

Cada curva de nível reúne as combinações de `tv` e `radio` que o modelo prevê com a mesma venda; a superfície cresce para a direita e para cima, do mesmo jeito que os dois coeficientes positivos calculados acima já anunciavam.

### O que a superfície plana ainda erra

O modelo com `tv` e `radio` explica 0,8972 da variância de vendas, mas sobra no resíduo um padrão que essa superfície plana não captura.

In [ ]:
df_resid = pd.DataFrame({
    "previsto": yhat_tv_radio,
    "residuo": y.to_numpy() - yhat_tv_radio,
})
df_resid["terco"] = pd.qcut(df_resid["previsto"], 3, labels=["baixo", "medio", "alto"])

medias_por_terco = df_resid.groupby("terco", observed=True)[["previsto", "residuo"]].mean()
medias_por_terco.round(4)

Dividindo os duzentos mercados em três grupos pelo valor previsto — o terço com a previsão mais baixa, o do meio, o mais alto —, o resíduo médio não fica perto de zero nos três grupos: 0,4307 no terço mais baixo, -0,8077 no do meio, e de volta a 0,3650 no mais alto. O sinal do erro típico muda com a faixa de previsão, em vez de se espalhar ao acaso ao redor de zero — um padrão que uma superfície plana, por definição, não reproduz.

In [ ]:
# Figura: Resíduo contra o valor previsto, no modelo de tv e radio. A linha tracejada marca resíduo zero; os três marcadores maiores são a média de cada terço de previsão — abaixo de zero no meio, acima nas duas pontas.
fig, ax = plt.subplots()
ax.axhline(0, color="C2", linewidth=1, linestyle="--")
ax.scatter(df_resid["previsto"], df_resid["residuo"], color="C0", s=14, alpha=0.6)
ax.plot(
    medias_por_terco["previsto"],
    medias_por_terco["residuo"],
    color="C1",
    linewidth=2,
    marker="o",
    markersize=7,
)
ax.set_xlabel("vendas prevista (mil unidades)")
ax.set_ylabel("resíduo (vendas observada − prevista)")
plt.tight_layout()
plt.show()

Essa curvatura — negativa no meio da faixa de previsão, positiva nas duas pontas — é o sinal de que `tv` e `radio` não agem de forma independente sobre vendas. A seção 8.5 retoma este mesmo modelo para mostrar o que muda ao deixar as duas mídias interagirem.

## Preditores Qualitativos

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.1 de James et al. (2023).

Um coeficiente de regressão multiplica um número por outro, $\beta_j \cdot x_j$. Todo preditor usado até aqui neste capítulo era numérico — dólares, unidades, anos. `Credit`, o conjunto que esta seção introduz, tem quatro colunas que não são: `imovel_proprio`, `estudante` e `casado` valem `sim` ou `não`; `regiao` vale `Leste`, `Sul` ou `Oeste`. Nenhuma delas multiplica coeficiente nenhum do jeito que está — e é essa a pergunta que a seção resolve.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

plt.style.use("estilo-figuras.mplstyle")

In [ ]:
credit = pd.read_csv("dados/Credit.csv")
credit.shape, credit.columns.tolist()

São 400 clientes e onze colunas: `renda`, `limite`, `pontuacao`, `cartoes`, `idade` e `escolaridade`, numéricas; `imovel_proprio`, `estudante` e `casado`, `sim`/`não`; `regiao`, com três categorias; e `saldo` — a dívida média no cartão de crédito de cada cliente, em dólares, a resposta que o resto da seção tenta prever a partir das quatro colunas categóricas.

### O saldo médio, por ser estudante ou não

A pergunta mais simples é a de duas categorias: quem é estudante carrega, em média, saldo diferente de quem não é? Uma variável indicadora responde trocando a categoria por um número.

> **🔷 Conceito**
>
> Uma **variável indicadora** (ou *dummy*) troca uma categoria por 0 ou 1: vale 1 quando a observação está no nível escolhido, 0 nos demais. Um preditor qualitativo com $k$ níveis entra no modelo como $k - 1$ indicadoras, nunca $k$ — a categoria que sobra sem indicadora própria é a **base**, e toda previsão para ela sai só do intercepto.

In [ ]:
indicadora_estudante = pd.get_dummies(credit[["estudante"]], drop_first=True)
indicadora_estudante.columns.tolist()

`drop_first=True` descarta a primeira categoria em ordem alfabética — `não` vem antes de `sim` — e fica só `estudante_sim`, que vale `True` para quem é estudante e `False` para quem não é. `não` é a base: toda observação sem a indicadora marcada pertence a ela.

In [ ]:
modelo_estudante = LinearRegression().fit(indicadora_estudante, credit["saldo"])
intercepto_estudante = round(float(modelo_estudante.intercept_), 2)
coef_estudante_sim = round(float(modelo_estudante.coef_[0]), 2)

intercepto_estudante, coef_estudante_sim

In [ ]:
medias_estudante = credit.groupby("estudante")["saldo"].mean()
media_nao_estudante = round(float(medias_estudante["não"]), 2)
media_sim_estudante = round(float(medias_estudante["sim"]), 2)
diferenca_medias_estudante = round(media_sim_estudante - media_nao_estudante, 2)
estudantes_devem_mais = bool(diferenca_medias_estudante > 0)

media_nao_estudante, media_sim_estudante, diferenca_medias_estudante, estudantes_devem_mais

In [ ]:
intercepto_bate_com_media = intercepto_estudante == media_nao_estudante
coeficiente_bate_com_diferenca = coef_estudante_sim == diferenca_medias_estudante

intercepto_bate_com_media, coeficiente_bate_com_diferenca

O modelo ajustado só com essa indicadora dá um intercepto de 480,37 e um coeficiente de 396,46. São os mesmos dois números que saem direto do dado, sem ajustar modelo nenhum: 480,37 é o saldo médio de quem não é estudante, e 396,46 é a diferença entre essa média e a de quem é, 876,83 (876,83 − 480,37 = 396,46). `intercepto_bate_com_media` e `coeficiente_bate_com_diferenca` confirmam a igualdade nos dois casos. `estudantes_devem_mais` confirma que quem é estudante carrega, em média, mais saldo que quem não é.

### A base é uma escolha, e a previsão não sabe disso

Trocar qual categoria fica sem indicadora — a base — é uma escolha de quem ajusta o modelo, não um fato sobre o dado. `get_dummies` descarta, por padrão, a primeira categoria em ordem alfabética; para inverter, basta listar as categorias na ordem desejada antes de gerar a indicadora.

In [ ]:
estudante_ordenado = pd.Categorical(credit["estudante"], categories=["sim", "não"])
indicadora_estudante_b = pd.get_dummies(pd.DataFrame({"estudante": estudante_ordenado}), drop_first=True)
indicadora_estudante_b.columns.tolist()

In [ ]:
modelo_estudante_b = LinearRegression().fit(indicadora_estudante_b, credit["saldo"])
intercepto_estudante_b = round(float(modelo_estudante_b.intercept_), 2)
coef_estudante_nao = round(float(modelo_estudante_b.coef_[0]), 2)

intercepto_estudante_b, coef_estudante_nao

Com `sim` como base, o intercepto salta para 876,83 — a mesma média de quem é estudante de antes — e o coeficiente muda de sinal, para -396,46: a mesma diferença de antes, contada na direção oposta.

In [ ]:
previsao_original = modelo_estudante.predict(indicadora_estudante)
previsao_invertida = modelo_estudante_b.predict(indicadora_estudante_b)
previsoes_identicas = bool(np.allclose(previsao_original, previsao_invertida))

previsoes_identicas

`previsoes_identicas` confirma que, apesar do intercepto e do coeficiente mudarem, as previsões para os 400 clientes são as mesmas nos dois ajustes — a escolha da base muda a leitura dos coeficientes, não o que o modelo prevê.

### Região: quando a categoria tem mais de dois nomes

`regiao` tem três categorias — `Leste`, `Sul` e `Oeste` —, não duas. Uma única indicadora não dá conta: sobra uma categoria sem representação. A regra do conceito acima vale de novo, com $k = 3$: entram $k - 1 = 2$ indicadoras.

In [ ]:
indicadora_regiao = pd.get_dummies(credit[["regiao"]], drop_first=True)
indicadora_regiao.columns.tolist()

Duas indicadoras, `regiao_Oeste` e `regiao_Sul`; `Leste` fica sem indicadora própria e é a base. Uma terceira indicadora, para `Leste`, seria redundante: quem não é `Oeste` nem `Sul` só pode ser `Leste`, e o valor dela já está determinado pelas outras duas.

In [ ]:
modelo_regiao = LinearRegression().fit(indicadora_regiao, credit["saldo"])
coeficientes_regiao = dict(zip(modelo_regiao.feature_names_in_, modelo_regiao.coef_))
coef_oeste = round(float(coeficientes_regiao["regiao_Oeste"]), 2)
coef_sul = round(float(coeficientes_regiao["regiao_Sul"]), 2)
intercepto_regiao = round(float(modelo_regiao.intercept_), 2)

intercepto_regiao, coef_oeste, coef_sul

In [ ]:
medias_regiao = credit.groupby("regiao")["saldo"].mean()
media_leste = round(float(medias_regiao["Leste"]), 2)
media_oeste = round(float(medias_regiao["Oeste"]), 2)
media_sul = round(float(medias_regiao["Sul"]), 2)
leste_tem_a_maior_media = bool(medias_regiao.idxmax() == "Leste")

media_leste, media_oeste, media_sul, leste_tem_a_maior_media

O intercepto, 531,0, é o saldo médio de quem mora no Leste — a base. Os coeficientes leem-se como diferenças contra essa base: -18,69 para `Oeste` e -12,5 para `Sul`, o mesmo que a tabela de médias mostra: 512,31 − 531,0 = -18,69 e 518,5 − 531,0 = -12,5. `leste_tem_a_maior_media` confirma que, das três regiões, é no Leste que o saldo médio é o mais alto.

### O que o modelo prevê: a média do grupo, e só ela

Com um único preditor qualitativo, o modelo não tem outra informação para usar: toda previsão é a média do grupo a que a observação pertence, e nada mais fino que isso. Para `estudante`, só existem duas previsões possíveis, 480,37 e 876,83; para `regiao`, só três, 531,0, 512,31 e 518,5.

In [ ]:
# Figura: Saldo de cada um dos 400 clientes, por estudante (esquerda) e por região (direita). Os pontos têm um deslocamento horizontal aleatório só para não empilhar; o traço laranja marca a média de cada grupo — a mesma que o intercepto e os coeficientes reproduzem.
rng = np.random.default_rng(8)

fig, (ax_estudante, ax_regiao) = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)

paineis = [
    (ax_estudante, "estudante", ["não", "sim"], medias_estudante, "estudante"),
    (ax_regiao, "regiao", ["Leste", "Sul", "Oeste"], medias_regiao, "região"),
]
for eixo, coluna, ordem, medias, rotulo in paineis:
    for posicao, categoria in enumerate(ordem):
        valores = credit.loc[credit[coluna] == categoria, "saldo"]
        deslocamento = rng.uniform(-0.15, 0.15, size=len(valores))
        eixo.scatter(posicao + deslocamento, valores, s=12, alpha=0.4, color="C0")
        eixo.plot([posicao - 0.22, posicao + 0.22], [medias[categoria]] * 2, color="C1", linewidth=3)
    eixo.set_xticks(range(len(ordem)))
    eixo.set_xticklabels(ordem)
    eixo.set_xlabel(rotulo)

ax_estudante.set_ylabel("saldo (dólares)")
plt.tight_layout()
plt.show()

O traço laranja fica na mesma altura para todo ponto do grupo: é essa reta plana, por categoria, que um preditor qualitativo sozinho consegue desenhar.

## Interação e Termos Não Lineares

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.2 de James et al. (2023).

A leitura que a seção 8.3 instalou — cada coeficiente mede o efeito do seu preditor mantendo os demais fixos — carrega duas suposições que até aqui ficaram sem exame. A primeira é que o efeito de um preditor não muda com o nível dos outros; a segunda é que esse efeito é constante ao longo de toda a faixa do preditor, ou seja, que a relação é uma reta. As duas têm nome — aditividade e linearidade — e as duas podem ser testadas em vez de assumidas. Esta seção testa as duas: primeiro solta a aditividade entre `tv` e `radio`, depois solta a linearidade entre `potencia` e `milhas_por_galao`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("estilo-figuras.mplstyle")

### O modelo aditivo assume demais

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
y_vendas = propaganda["vendas"]

X_aditivo = propaganda[["tv", "radio"]]
modelo_aditivo = LinearRegression().fit(X_aditivo, y_vendas)
r2_aditivo = float(r2_score(y_vendas, modelo_aditivo.predict(X_aditivo)))

round(r2_aditivo, 4)

O modelo aditivo de `tv` e `radio` — o mesmo que a seção 8.3 fechou apontando um padrão no resíduo — explica 0,8972 da variância de vendas. "Aditivo" é a palavra técnica para uma suposição específica: o quanto vendas sobe para cada dólar a mais em `tv` é sempre o mesmo $\hat\beta_1$, não importa quanto se gasta em `radio` — as duas mídias contribuem em paralelo, sem uma alterar o efeito da outra. É uma suposição conveniente, e nada nela garante que seja verdadeira.

### Uma interação entre as duas mídias

> **🔷 Conceito**
>
> Um **termo de interação** soma ao modelo o produto de dois preditores, $X_1 \cdot X_2$. Ele relaxa a aditividade: reescrevendo $\beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_1 X_2$ em função de $X_1$, o coeficiente que multiplica $X_1$ deixa de ser a constante $\beta_1$ e passa a ser $\beta_1 + \beta_3 X_2$ — um valor que muda com $X_2$. O efeito de um preditor passa a depender do nível do outro.

In [ ]:
propaganda["tv_radio"] = propaganda["tv"] * propaganda["radio"]

X_interacao = propaganda[["tv", "radio", "tv_radio"]]
modelo_interacao = LinearRegression().fit(X_interacao, y_vendas)
r2_interacao = float(r2_score(y_vendas, modelo_interacao.predict(X_interacao)))

round(r2_interacao, 4)

In [ ]:
comparacao_interacao = pd.DataFrame(
    {"R²": [r2_aditivo, r2_interacao]},
    index=["aditivo (tv + radio)", "com interação (tv + radio + tv×radio)"],
).round(4)
interacao_supera_aditivo = bool(r2_interacao > r2_aditivo)
pct_restante_explicado = round((r2_interacao - r2_aditivo) / (1 - r2_aditivo) * 100, 1)

comparacao_interacao, interacao_supera_aditivo, pct_restante_explicado

`interacao_supera_aditivo` confirma que somar a coluna `tv_radio` sobe o R² — de 0,8972 para 0,9678. Da variância que ainda sobrava depois do modelo aditivo, 68,7% foi explicada pela interação.

In [ ]:
coef_interacao = pd.Series(modelo_interacao.coef_, index=modelo_interacao.feature_names_in_)
coef_interacao_mil = (coef_interacao * 1000).round(2)

coef_interacao.round(4), coef_interacao_mil

Os três coeficientes ficam positivos: 0,0191 para `tv`, 0,0289 para `radio`, 0,0011 para `tv_radio`. Postos em dólares por mil — 19,10, 28,86 e 1,09 — eles compõem a leitura que o conceito acima descreveu em símbolos: cada mil dólares a mais em `tv`, mantendo `radio` fixo, está associado a (19,10 + 1,09 × radio) unidades a mais de venda; cada mil dólares a mais em `radio`, mantendo `tv` fixo, a (28,86 + 1,09 × tv) unidades a mais. O efeito de uma mídia cresce com o nível da outra — é essa a "sinergia" que o termo de interação captura, e é ela que falta ao modelo aditivo.

> **🔷 Conceito**
>
> O **princípio da hierarquia** diz que, se um termo de interação entra no modelo, os dois termos principais que o compõem entram junto — mesmo que o coeficiente de algum deles pareça pequeno diante do outro. A razão: `tv_radio` está correlacionada com `tv` e com `radio`, então excluir um dos dois muda o que a interação está de fato medindo. `tv` e `radio` continuam no modelo `tv + radio + tv_radio` acima por essa regra, não por terem coeficiente grande.

### Interação com um preditor qualitativo

A interação não pede que os dois preditores sejam numéricos. `Credit`, com a indicadora que a seção 8.4 introduziu para `estudante`, permite a mesma pergunta entre uma variável numérica e uma categórica: o efeito de `renda` sobre `saldo` é o mesmo para quem é estudante e para quem não é?

In [ ]:
credit = pd.read_csv("dados/Credit.csv")
indicadora_estudante = pd.get_dummies(credit[["estudante"]], drop_first=True)
credit["estudante_sim"] = indicadora_estudante["estudante_sim"].astype(int)
y_saldo = credit["saldo"]

credit.shape

In [ ]:
X_sem_interacao = credit[["renda", "estudante_sim"]]
modelo_credit_sem = LinearRegression().fit(X_sem_interacao, y_saldo)
coef_credit_sem = pd.Series(modelo_credit_sem.coef_, index=modelo_credit_sem.feature_names_in_)

coef_credit_sem.round(2), round(float(modelo_credit_sem.intercept_), 2)

In [ ]:
credit["renda_estudante"] = credit["renda"] * credit["estudante_sim"]

X_com_interacao = credit[["renda", "estudante_sim", "renda_estudante"]]
modelo_credit_com = LinearRegression().fit(X_com_interacao, y_saldo)
coef_credit_com = pd.Series(modelo_credit_com.coef_, index=modelo_credit_com.feature_names_in_)
inclinacao_estudante_com_interacao = round(float(coef_credit_com["renda"] + coef_credit_com["renda_estudante"]), 2)

coef_credit_com.round(2), round(float(modelo_credit_com.intercept_), 2), inclinacao_estudante_com_interacao

Sem interação, `renda` vale 5,98 (dólares de saldo a mais por mil dólares de renda, mantendo o status de estudante fixo) e `estudante_sim` vale 382,67 — a mesma diferença entre estudantes e não estudantes, não importa a renda. Com a interação, `renda` sobe para 6,22, `estudante_sim` para 476,68, e `renda_estudante` sai negativo, -2,00: a inclinação de quem é estudante deixa de ser a de quem não é, e passa a ser 4,22 — mais baixa que a dos não estudantes. É exatamente a leitura que o modelo sem interação não permitia: ali as duas retas tinham a mesma inclinação por construção; aqui elas não têm mais.

In [ ]:
# Figura: Saldo contra renda em Credit, para não estudantes e estudantes. Esquerda: modelo sem interação — as duas retas são paralelas. Direita: modelo com interação renda×estudante — as inclinações diferem. Os pontos são os 400 clientes observados.
grade_renda = np.linspace(credit["renda"].min(), credit["renda"].max(), 100)
grupos = [(0, "C0", "não"), (1, "C1", "sim")]

fig, (ax_sem, ax_com) = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)

for valor_grupo, cor, rotulo in grupos:
    pontos = credit[credit["estudante_sim"] == valor_grupo]
    ax_sem.scatter(pontos["renda"], pontos["saldo"], s=10, alpha=0.35, color=cor)
    ax_com.scatter(pontos["renda"], pontos["saldo"], s=10, alpha=0.35, color=cor)

    grade_sem = pd.DataFrame({"renda": grade_renda, "estudante_sim": valor_grupo})
    ax_sem.plot(grade_renda, modelo_credit_sem.predict(grade_sem), color=cor, linewidth=2.4, label=rotulo)

    grade_com = pd.DataFrame(
        {"renda": grade_renda, "estudante_sim": valor_grupo, "renda_estudante": grade_renda * valor_grupo}
    )
    ax_com.plot(grade_renda, modelo_credit_com.predict(grade_com), color=cor, linewidth=2.4, label=rotulo)

ax_sem.set_title("sem interação")
ax_com.set_title("com interação")
ax_sem.set_xlabel("renda (milhares de dólares)")
ax_com.set_xlabel("renda (milhares de dólares)")
ax_sem.set_ylabel("saldo (dólares)")
ax_com.legend(title="estudante")
plt.tight_layout()
plt.show()

O painel esquerdo mostra o que a suposição sem interação força: duas retas com a mesma inclinação, deslocadas por 382,67 em saldo, em qualquer renda. O painel direito mostra o que a interação libera: a reta de quem é estudante nasce mais alta — 476,68 acima da de quem não é, em `renda` zero — mas sobe mais devagar, e a distância entre as duas retas encolhe à medida que a renda cresce.

### Potência não anda em linha reta com milhas por galão

`Auto` traz uma armadilha de tipo antes de qualquer regressão: a coluna `potencia`, que deveria ser número, chega como texto em cinco linhas.

In [ ]:
auto = pd.read_csv("dados/Auto.csv")
n_auto_bruto = len(auto)
n_potencia_interrogacao = int((auto["potencia"] == "?").sum())

auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
auto = auto.dropna(subset=["potencia"]).reset_index(drop=True)
n_auto_limpo = len(auto)

n_auto_bruto, n_potencia_interrogacao, n_auto_limpo

São 397 carros no arquivo, e cinco trazem `potencia` como o texto `"?"` em vez de um número — é por isso que a coluna não chega como `float` de saída do `read_csv`. `pd.to_numeric(..., errors="coerce")` troca cada `"?"` por `NaN` em vez de derrubar a leitura inteira, e `dropna` descarta essas cinco linhas: restam 392 carros para o que segue.

In [ ]:
X_potencia = auto[["potencia"]]
y_milhas = auto["milhas_por_galao"]

modelos_grau = {}
r2_grau = {}
for grau in (1, 2, 5):
    modelo = make_pipeline(StandardScaler(), PolynomialFeatures(degree=grau, include_bias=False), LinearRegression())
    modelo.fit(X_potencia, y_milhas)
    modelos_grau[grau] = modelo
    r2_grau[grau] = float(r2_score(y_milhas, modelo.predict(X_potencia)))

comparacao_graus = pd.DataFrame(
    {"R²": [r2_grau[1], r2_grau[2], r2_grau[5]]}, index=["grau 1 (reta)", "grau 2", "grau 5"]
).round(4)
melhora_grau2_sobre_grau1 = round(r2_grau[2] - r2_grau[1], 4)
melhora_grau5_sobre_grau2 = round(r2_grau[5] - r2_grau[2], 4)

comparacao_graus, melhora_grau2_sobre_grau1, melhora_grau5_sobre_grau2

`StandardScaler` entra antes de `PolynomialFeatures` no `Pipeline` porque `potencia` chega a valores acima de 200, e a quinta potência de um número desse tamanho deixa as colunas do modelo em escalas tão distintas que a solução de mínimos quadrados perde precisão — centralizar e normalizar antes de elevar à potência evita o problema, sem mudar a curva que sai no fim.

A reta explica 0,6059 da variância de milhas por galão, e a parábola de grau 2 sobe para 0,6876. Essas quatro casas não bastam para refazer a melhora à mão — calculada sobre os R² completos, sem arredondar antes, a melhora de ir da reta para o grau 2 é 0,0816. Do grau 2 para o grau 5 o R² sobe só até 0,6967, uma melhora de 0,0092 pelo mesmo cálculo: três parâmetros a mais quase não mudam o quanto o modelo explica dos 392 carros que ele já viu.

In [ ]:
# Figura: milhas_por_galao contra potencia em Auto, com a reta (grau 1), a parábola (grau 2) e o polinômio de grau 5 ajustados sobre os 392 carros com potencia numérica. Na ponta direita da faixa, a curva de grau 5 volta a subir; a de grau 2 quase não sai do lugar.
grade_potencia = pd.DataFrame({"potencia": np.linspace(X_potencia["potencia"].min(), X_potencia["potencia"].max(), 300)})
cores_grau = {1: "C1", 2: "C0", 5: "C2"}
rotulos_grau = {1: "grau 1", 2: "grau 2", 5: "grau 5"}

fig, ax = plt.subplots()
ax.scatter(auto["potencia"], auto["milhas_por_galao"], s=14, alpha=0.35, color="#6C757D")
for grau in (1, 2, 5):
    previsao = modelos_grau[grau].predict(grade_potencia)
    ax.plot(grade_potencia["potencia"], previsao, color=cores_grau[grau], linewidth=2.2, label=rotulos_grau[grau])

ax.set_xlabel("potência (hp)")
ax.set_ylabel("milhas por galão")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
previsao_grau2 = modelos_grau[2].predict(grade_potencia)
previsao_grau5 = modelos_grau[5].predict(grade_potencia)
depois_de_150 = (grade_potencia["potencia"] > 150).to_numpy()

indice_minimo_grau2 = np.argmin(np.where(depois_de_150, previsao_grau2, np.inf))
indice_minimo_grau5 = np.argmin(np.where(depois_de_150, previsao_grau5, np.inf))

subida_grau2 = float(previsao_grau2[-1] - previsao_grau2[indice_minimo_grau2])
subida_grau5 = float(previsao_grau5[-1] - previsao_grau5[indice_minimo_grau5])
grau5_ondula_mais = bool(subida_grau5 > subida_grau2)

round(subida_grau2, 2), round(subida_grau5, 2), grau5_ondula_mais

`grau5_ondula_mais` confirma o que a figura mostra: depois do ponto mais baixo de cada curva na metade final da faixa de potência, a de grau 2 sobe 2,03 milhas por galão até a ponta direita — quase reta —, enquanto a de grau 5 sobe 5,19, mais que o dobro, desenhando a curvatura extra que o R² por si só não deixava ver: um ganho de 0,0092 no ajuste veio acompanhado de uma curva que se dobra de volta para cima nas pontas, em vez de continuar a tendência de queda que os 392 pontos sugerem.

### Ainda é regressão linear

Os três modelos de `potencia` acima — reta, parábola, quinto grau — saíram do mesmo estimador que abriu o capítulo: `LinearRegression`, chamado dentro de um `Pipeline` que só troca as colunas de entrada, nunca o estimador. Um modelo com `potencia²` e `potencia⁵` continua sendo **regressão linear** porque "linear" descreve como o modelo soma seus parâmetros $\beta$ — cada um multiplicando uma coluna, todos somados —, não o formato da curva que esses parâmetros produzem quando a coluna multiplicada é ela mesma uma potência do preditor original. A curva pode dobrar; a soma por trás dela continua linear nos $\beta$.

## Outliers, Alavancagem e Colinearidade

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.3 de James et al. (2023).

A seção 8.5 comparou reta, parábola e polinômio de grau 5 ajustados a `potencia`, olhando tanto para o R² de cada um quanto para o quanto a curva de grau 5 ondula nas pontas. Nenhuma dessas duas medidas, porém, diz se o erro que sobra num ajuste tem um padrão que uma curva melhor teria capturado, se um único carro está puxando o ajuste sozinho, ou se dois preditores andam tão juntos que seus coeficientes deixaram de significar algo isolado. Esta seção olha para essas perguntas — o padrão que sobra no resíduo, o outlier, a alavancagem e a colinearidade —, e o instrumento comum às quatro é o gráfico.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.style.use("estilo-figuras.mplstyle")

auto = pd.read_csv("dados/Auto.csv")
auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
auto = auto.dropna(subset=["potencia"]).reset_index(drop=True)
auto["potencia2"] = auto["potencia"] ** 2

### O resíduo contra o previsto: onde a reta ainda erra

In [ ]:
y_milhas = auto["milhas_por_galao"].to_numpy()
n = len(auto)

X1 = auto[["potencia"]]
modelo1 = LinearRegression().fit(X1, y_milhas)
previsto1 = modelo1.predict(X1)
residuo1 = y_milhas - previsto1
rse1 = float(np.sqrt((residuo1**2).sum() / (n - 1 - 1)))
r2_1 = float(r2_score(y_milhas, previsto1))

X2 = auto[["potencia", "potencia2"]]
modelo2 = LinearRegression().fit(X2, y_milhas)
previsto2 = modelo2.predict(X2)
residuo2 = y_milhas - previsto2
rse2 = float(np.sqrt((residuo2**2).sum() / (n - 2 - 1)))
r2_2 = float(r2_score(y_milhas, previsto2))

n, round(r2_1, 2), round(rse1, 2), round(r2_2, 2), round(rse2, 2)

Restam os 392 carros que a 8.5 já deixou sem o `"?"` de `potencia`. Ajustada só contra `potencia`, a reta explica 0,61 da variância de `milhas_por_galao`, com erro típico de 4,91 milhas por galão; somando `potencia²`, o R² sobe para 0,69 e o erro cai para 4,37. Essas duas medidas dizem o quanto o modelo erra em média — não dizem se o erro que sobra tem um padrão que a reta deveria ter capturado. Para isso serve o resíduo contra o previsto, não a resposta bruta.

In [ ]:
# Figura: Resíduo contra o valor previsto, para o ajuste de milhas_por_galao sobre potencia (esquerda) e sobre potencia e potencia² (direita), nos 392 carros com potencia numérica. A linha laranja liga a média do resíduo em cada uma de oito faixas do valor previsto: à esquerda ela desce e sobe de novo, desenhando o U que indica não linearidade; à direita as médias ficam bem mais próximas de zero — embora a faixa mais alta ainda chegue a -2,94 —, sem o formato de U.
def medias_por_faixa(previsto, residuo, n_faixas=8):
    faixa = pd.cut(previsto, n_faixas)
    df = pd.DataFrame({"previsto": previsto, "residuo": residuo, "faixa": faixa})
    return df.groupby("faixa", observed=True).agg(
        previsto_medio=("previsto", "mean"), residuo_medio=("residuo", "mean")
    )

medias1 = medias_por_faixa(previsto1, residuo1)
medias2 = medias_por_faixa(previsto2, residuo2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)
for ax, previsto, residuo, medias, titulo in [
    (ax1, previsto1, residuo1, medias1, "grau 1"),
    (ax2, previsto2, residuo2, medias2, "grau 2"),
]:
    ax.axhline(0, color="C2", linewidth=1, linestyle="--")
    ax.scatter(previsto, residuo, color="C0", s=14, alpha=0.4)
    ax.plot(
        medias["previsto_medio"], medias["residuo_medio"],
        color="C1", linewidth=2, marker="o", markersize=6,
    )
    ax.set_title(titulo)
    ax.set_xlabel("previsto (milhas por galão)")
ax1.set_ylabel("resíduo")
plt.tight_layout()
plt.show()

In [ ]:
desvio_medias1 = float(medias1["residuo_medio"].std(ddof=0))
desvio_medias2 = float(medias2["residuo_medio"].std(ddof=0))

round(desvio_medias1, 2), round(desvio_medias2, 2)

O formato conta a história: à esquerda, as oito médias por faixa desenham um U — alta nas duas pontas, negativa no meio —, e o desvio-padrão entre elas é 3,15; à direita, `potencia²` já resolveu boa parte desse padrão, e o mesmo desvio-padrão cai para 1,38. Sobra menos estrutura para o modelo explicar depois de somar o termo quadrático.

### O ponto que o modelo erra sozinho: outlier

> **🔷 Conceito**
>
> O **resíduo padronizado** de uma observação é o resíduo dividido pelo erro-padrão da regressão (RSE) daquele ajuste, $e_i / \text{RSE}$. O ISLP usa uma versão mais refinada, que divide cada resíduo pelo seu próprio erro-padrão estimado — o **resíduo estudentizado** — e aponta valores acima de 3 em módulo como possíveis outliers: um ponto cuja resposta observada está longe da prevista, mesmo depois de descontar a escala típica do erro.

In [ ]:
residuo_padronizado = residuo2 / rse2
idx_pior = int(np.argmax(np.abs(residuo_padronizado)))
nome_pior = auto.loc[idx_pior, "nome"]
pior_valor = float(residuo_padronizado[idx_pior])
n_acima_de_3 = int((np.abs(residuo_padronizado) > 3).sum())
segundo_pior_valor = float(np.sort(np.abs(residuo_padronizado))[-2])

nome_pior, round(pior_valor, 2), n_acima_de_3, round(segundo_pior_valor, 2)

O maior resíduo padronizado, em valor absoluto, é o do `datsun 280-zx`: 3,63, acima do corte de 3 que o ISLP usa para sinalizar um possível outlier. Mas ele não está isolado do resto: cinco dos 392 carros passam de 3 em módulo, e o segundo colocado, 3,38, fica a menos de 0,3 de distância do primeiro — um candidato a outlier, não um ponto isolado do resto como no exemplo do livro-texto.

In [ ]:
mascara_sem_pior = np.ones(n, dtype=bool)
mascara_sem_pior[idx_pior] = False

modelo2_sem_pior = LinearRegression().fit(X2[mascara_sem_pior], y_milhas[mascara_sem_pior])
previsto2_sem_pior = modelo2_sem_pior.predict(X2[mascara_sem_pior])
r2_2_sem_pior = float(r2_score(y_milhas[mascara_sem_pior], previsto2_sem_pior))

coef_potencia_antes = float(modelo2.coef_[0])
coef_potencia_depois = float(modelo2_sem_pior.coef_[0])

round(r2_2, 2), round(r2_2_sem_pior, 2), round(coef_potencia_antes, 2), round(coef_potencia_depois, 2)

Tirar só esse carro e reajustar muda pouco: o R² sobe de 0,69 para 0,70, e o coeficiente de `potencia` continua em -0,47 nas duas casas decimais. É um outlier fraco — desvia bastante em y, mas não desloca o ajuste porque o seu valor de `potencia` não é incomum: como a próxima seção mede, a alavancagem desse carro fica abaixo da média das 392 observações. Um outlier de alta alavancagem desloca o ajuste; um outlier de baixa alavancagem, como este, não.

### Alavancagem: quando o exagero está no eixo x

> **🔷 Conceito**
>
> A **alavancagem** $h_{ii}$ de uma observação mede o quão incomum é o seu valor de preditor — não o quão longe a resposta observada fica da prevista. Em regressão simples, o ISLP dá a fórmula fechada, a equação (3.37) do livro-texto:
>
> $$
> h_i = \frac{1}{n} + \frac{(x_i - \bar{x})^2}{\displaystyle\sum_{i'=1}^{n} (x_{i'} - \bar{x})^2}.
> $$
>
> $h_i$ cresce com a distância de $x_i$ à média. Com mais de um preditor, a extensão é a **matriz chapéu** (*hat matrix*):
>
> $$
> H = X (X^\top X)^{-1} X^\top, \qquad h_{ii} = H_{ii},
> $$
>
> em que $X$ tem uma coluna de 1 para o intercepto e uma coluna para cada preditor. $h_{ii}$ está sempre entre $1/n$ e 1, e sua média sobre as $n$ observações é sempre $(p+1)/n$, com $p$ preditores.

In [ ]:
X_design = np.column_stack([np.ones(n), X2.to_numpy()])
H = X_design @ np.linalg.inv(X_design.T @ X_design) @ X_design.T
alavancagem = np.diag(H)

p = X2.shape[1]
media_alavancagem = float(alavancagem.mean())
alavancagem_esperada = (p + 1) / n
alavancagem_bate_com_formula = bool(np.isclose(media_alavancagem, alavancagem_esperada))

round(media_alavancagem, 4), round(alavancagem_esperada, 4), alavancagem_bate_com_formula

A média das 392 alavancagens, 0,0077, bate com $(p+1)/n$ para $p=2$ preditores — `alavancagem_bate_com_formula` confirma `True`. Não é um acaso deste ajuste em particular; é o que a fórmula promete para qualquer regressão linear.

In [ ]:
limiar_alavancagem = 3 * media_alavancagem
alta_alavancagem = alavancagem > limiar_alavancagem
e_outlier = np.abs(residuo_padronizado) > 3

n_alta_alavancagem = int(alta_alavancagem.sum())
maior_residuo_entre_alta_alavancagem = float(np.max(np.abs(residuo_padronizado[alta_alavancagem])))
maior_alavancagem_entre_outliers = float(np.max(alavancagem[e_outlier]))
nenhum_carro_e_os_dois = bool(not np.any(alta_alavancagem & e_outlier))

(
    n_alta_alavancagem,
    round(maior_residuo_entre_alta_alavancagem, 2),
    round(maior_alavancagem_entre_outliers, 4),
    nenhum_carro_e_os_dois,
)

Um critério usual para "alavancagem alta" é passar do triplo da média: dos 15 carros que passam desse limiar, o maior resíduo padronizado em módulo é 2,76 — abaixo do corte de outlier; dos 5 carros com resíduo padronizado acima de 3, a maior alavancagem é 0,0076 — abaixo da própria média. `nenhum_carro_e_os_dois` confirma `True`: nenhum carro deste conjunto combina os dois problemas ao mesmo tempo, o que a figura a seguir mostra.

In [ ]:
# Figura: Resíduo padronizado contra alavancagem, para os 392 carros no ajuste com potencia e potencia². O ponto laranja é o pior resíduo (datsun 280-zx); o ponto roxo é a maior alavancagem (pontiac grand prix). São problemas diferentes, e nenhum carro combina os dois.
idx_maior_alavancagem = int(np.argmax(alavancagem))
nome_maior_alavancagem = auto.loc[idx_maior_alavancagem, "nome"]

fig, ax = plt.subplots()
ax.axhline(0, color="C2", linewidth=1, linestyle="--")
ax.scatter(alavancagem, residuo_padronizado, color="C0", s=16, alpha=0.45)
ax.scatter([alavancagem[idx_pior]], [residuo_padronizado[idx_pior]], color="C1", s=55, zorder=3)
ax.annotate(
    nome_pior, xy=(alavancagem[idx_pior], residuo_padronizado[idx_pior]),
    xytext=(8, 4), textcoords="offset points", fontsize=8,
)
ax.scatter(
    [alavancagem[idx_maior_alavancagem]], [residuo_padronizado[idx_maior_alavancagem]],
    color="C3", s=55, zorder=3,
)
ax.annotate(
    nome_maior_alavancagem,
    xy=(alavancagem[idx_maior_alavancagem], residuo_padronizado[idx_maior_alavancagem]),
    xytext=(8, 4), textcoords="offset points", fontsize=8,
)
ax.set_xlabel("alavancagem")
ax.set_ylabel("resíduo padronizado")
plt.tight_layout()
plt.show()

In [ ]:
alavancagem_do_pior_residuo = float(alavancagem[idx_pior])
maior_alavancagem = float(alavancagem[idx_maior_alavancagem])
residuo_padronizado_da_maior_alavancagem = float(residuo_padronizado[idx_maior_alavancagem])
potencia_da_maior_alavancagem = float(auto.loc[idx_maior_alavancagem, "potencia"])
potencia_e_a_maior_do_conjunto = bool(potencia_da_maior_alavancagem == auto["potencia"].max())

(
    round(alavancagem_do_pior_residuo, 4),
    round(maior_alavancagem, 2),
    round(residuo_padronizado_da_maior_alavancagem, 2),
    potencia_da_maior_alavancagem,
    potencia_e_a_maior_do_conjunto,
)

O `datsun 280-zx`, o pior resíduo, tem alavancagem 0,0066 — abaixo da média 0,0077: seu problema é só em y, o valor previsto para a sua `potencia`. Do outro lado, o `pontiac grand prix` tem a maior alavancagem do conjunto, 0,09, e `potencia_e_a_maior_do_conjunto` confirma por quê: sua `potencia`, 230 hp, é a mais alta entre os 392 carros. Mas seu resíduo padronizado é só 0,28, longe do corte de 3 — os dois carros ilustram, cada um do seu lado, o que o chunk anterior já tinha confirmado para o conjunto inteiro.

### Colinearidade: quando dois preditores quase se confundem

In [ ]:
credit = pd.read_csv("dados/Credit.csv")
y_saldo = credit["saldo"].to_numpy().astype(float)

correlacoes_credit = credit[["limite", "pontuacao", "idade"]].corr()
correlacoes_credit.round(3)

`limite` e `pontuacao` correlacionam a 0,997 — quase 1, mas não exatamente, como o VIF adiante confirma —; `limite` e `idade`, a 0,10. Um sobe junto com o outro; o outro não tem relação visível com nenhum dos dois.

In [ ]:
X_idade_limite = credit[["idade", "limite"]]
modelo_idade_limite = LinearRegression().fit(X_idade_limite, y_saldo)

X_pontuacao_limite = credit[["pontuacao", "limite"]]
modelo_pontuacao_limite = LinearRegression().fit(X_pontuacao_limite, y_saldo)

coef_idade = float(modelo_idade_limite.coef_[0])
coef_limite_com_idade = float(modelo_idade_limite.coef_[1])
coef_pontuacao = float(modelo_pontuacao_limite.coef_[0])
coef_limite_com_pontuacao = float(modelo_pontuacao_limite.coef_[1])

round(coef_idade, 2), round(coef_limite_com_idade, 2), round(coef_pontuacao, 2), round(coef_limite_com_pontuacao, 2)

Com `idade`, o coeficiente de `limite` é 0,17 (o de `idade`, -2,29): mais um dólar de limite está associado a 0,17 dólar a mais de saldo, mantendo a idade fixa. Trocando `idade` por `pontuacao` — que anda junto com `limite` — o coeficiente de `limite` cai para 0,02, e o de `pontuacao` sobe para 2,20. `limite` não perdeu poder explicativo: a colinearidade só redistribuiu o crédito entre os dois preditores que se sobrepõem, e não há como separar, só olhando os coeficientes, quanto é de cada um.

> **🔷 Conceito**
>
> O **fator de inflação da variância** (VIF) de um preditor $X_j$ é
>
> $$
> \text{VIF}(\hat\beta_j) = \frac{1}{1 - R^2_{X_j \mid X_{-j}}},
> $$
>
> em que $R^2_{X_j \mid X_{-j}}$ é o R² da regressão de $X_j$ sobre todos os demais preditores — uma conta de R², não uma estatística de teste. VIF = 1 é ausência completa de colinearidade; o ISLP usa 5 ou 10 como referência para um VIF problemático.

In [ ]:
def vif(preditor, tabela, todos):
    outros = [c for c in todos if c != preditor]
    modelo_aux = LinearRegression().fit(tabela[outros], tabela[preditor])
    r2_aux = r2_score(tabela[preditor], modelo_aux.predict(tabela[outros]))
    return 1 / (1 - r2_aux)

preditores_vif = ["idade", "limite", "pontuacao"]
vif_idade = vif("idade", credit, preditores_vif)
vif_limite = vif("limite", credit, preditores_vif)
vif_pontuacao = vif("pontuacao", credit, preditores_vif)

round(vif_idade, 2), round(vif_limite, 2), round(vif_pontuacao, 2)

`idade` sai com VIF 1,01 — sem colinearidade —; `limite` e `pontuacao` saem com 160,59 e 160,67, muito acima da referência de 5 a 10 que o ISLP usa para um VIF problemático. A correlação já mostrava os dois andando juntos; o VIF diz o quanto isso infla a variância de cada coeficiente por causa disso.

A seção 8.1 mostrou que RSS, como função dos coeficientes, é um vale com um único fundo. Colinearidade não muda isso — ainda há um único mínimo —, mas muda o formato do vale ao redor dele.

In [ ]:
# Figura: Curvas de nível do RSS da regressão de saldo sobre dois preditores, em função dos coeficientes desses dois preditores — o intercepto fica fixo na média de saldo. Esquerda: idade e limite, que quase não se correlacionam — o vale é uma elipse fechada e comparativamente redonda. Direita: pontuacao e limite, que correlacionam a 0,997 — o vale vira uma calha comprida e estreita, na diagonal: muitos pares (β_pontuacao, β_limite) quase empatam em RSS. As escalas de β_limite dos dois painéis são diferentes — a calha da direita ocupa uma faixa bem mais larga do eixo —, então parecer mais fina não significa mais precisa: é a mesma largura relativa, numa escala maior. O ponto marca o par que minimiza RSS em cada ajuste.
def grade_rss(preditores, k=8, pontos=700, niveis_relativos=(0.02, 0.05, 0.1, 0.2, 0.4)):
    X = credit[preditores].to_numpy(dtype=float)
    Xc = X - X.mean(axis=0)
    yc = y_saldo - y_saldo.mean()
    M = Xc.T @ Xc
    M_inv = np.linalg.inv(M)
    beta_hat = np.linalg.lstsq(Xc, yc, rcond=None)[0]
    rss_min = float(((yc - Xc @ beta_hat) ** 2).sum())

    largura0 = np.sqrt(0.01 * rss_min * M_inv[0, 0])
    largura1 = np.sqrt(0.01 * rss_min * M_inv[1, 1])
    grade0 = np.linspace(beta_hat[0] - k * largura0, beta_hat[0] + k * largura0, pontos)
    grade1 = np.linspace(beta_hat[1] - k * largura1, beta_hat[1] + k * largura1, pontos)
    malha0, malha1 = np.meshgrid(grade0, grade1)
    d0, d1 = malha0 - beta_hat[0], malha1 - beta_hat[1]
    rss = rss_min + M[0, 0] * d0**2 + 2 * M[0, 1] * d0 * d1 + M[1, 1] * d1**2
    niveis = rss_min * (1 + np.array(niveis_relativos))
    return malha0, malha1, rss, niveis, beta_hat

larguras_das_curvas = [2.2, 1.8, 1.4, 1.0, 0.7]

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10, 4.6))
contornos_pares = []
for ax, preditores, rotulos in [
    (ax_a, ["idade", "limite"], ("coeficiente de idade", "coeficiente de limite")),
    (ax_b, ["pontuacao", "limite"], ("coeficiente de pontuacao", "coeficiente de limite")),
]:
    malha0, malha1, rss, niveis, beta_hat = grade_rss(preditores)
    contornos = ax.contour(malha0, malha1, rss, levels=niveis, colors="C0", linewidths=larguras_das_curvas)
    ax.scatter([beta_hat[0]], [beta_hat[1]], color="C1", s=35, zorder=3)
    ax.set_xlabel(rotulos[0])
    ax.set_ylabel(rotulos[1])
    contornos_pares.append(contornos)
plt.tight_layout()
plt.show()

In [ ]:
def conta_fechadas(contornos):
    total, fechadas = 0, 0
    for segmentos_do_nivel in contornos.allsegs:
        for segmento in segmentos_do_nivel:
            if len(segmento) == 0:
                continue
            total += 1
            if np.allclose(segmento[0], segmento[-1]):
                fechadas += 1
    return total, fechadas

total_a, fechadas_a = conta_fechadas(contornos_pares[0])
total_b, fechadas_b = conta_fechadas(contornos_pares[1])

total_a, fechadas_a, total_b, fechadas_b

As cinco curvas de cada painel fecham: `fechadas_a` e `fechadas_b` batem com `total_a` e `total_b` — 5 de 5 nos dois casos. Nenhuma é ladeira aberta se confundindo com vale, nem no par redondo nem no par esticado.

In [ ]:
def largura_beta_limite(preditores, eps):
    X = credit[preditores].to_numpy(dtype=float)
    Xc = X - X.mean(axis=0)
    yc = y_saldo - y_saldo.mean()
    M = Xc.T @ Xc
    M_inv = np.linalg.inv(M)
    beta_hat = np.linalg.lstsq(Xc, yc, rcond=None)[0]
    rss_min = float(((yc - Xc @ beta_hat) ** 2).sum())
    return 2 * float(np.sqrt(eps * rss_min * M_inv[1, 1]))

def razao_larguras(eps):
    return largura_beta_limite(["pontuacao", "limite"], eps) / largura_beta_limite(["idade", "limite"], eps)

largura_idade_limite = largura_beta_limite(["idade", "limite"], 0.01)
largura_pontuacao_limite = largura_beta_limite(["pontuacao", "limite"], 0.01)
razao_com_1_por_cento = razao_larguras(0.01)
razao_com_5_por_cento = razao_larguras(0.05)
razao_invariante_em_eps = bool(np.isclose(razao_com_1_por_cento, razao_com_5_por_cento))

round(largura_idade_limite, 2), round(largura_pontuacao_limite, 2), round(razao_com_1_por_cento, 2), razao_invariante_em_eps

A largura mede a calha diretamente: fixando o outro coeficiente no valor que melhor cabe, a faixa de $\beta_{\text{limite}}$ que ainda deixa RSS a até 1% do mínimo mede 0,02 no par (idade, limite) e 0,25 no par (pontuacao, limite) — 12,70 vezes mais larga. O limiar de 1% é só uma janela para medir; a razão entre as duas larguras não depende dele, porque as duas larguras crescem com $\sqrt{\varepsilon}$ e o $\varepsilon$ cancela na divisão — repetindo a conta com 5% em vez de 1%, `razao_invariante_em_eps` confirma `True`. É essa largura que a calha da figura desenha: quanto mais colinear o par, mais larga é a faixa de pares de coeficientes que o dado observado quase não distingue — e é isso que torna os coeficientes instáveis.

### Quatro problemas, um instrumento comum

Não linearidade, outlier, alavancagem e colinearidade não vêm sempre juntos. Neste capítulo, o U do resíduo linear tinha causa clara — faltava `potencia²` — e diminui bastante ao corrigi-la, com o desvio-padrão das médias por faixa caindo de 3,15 para 1,38; o outlier e a maior alavancagem recaíram sobre carros diferentes, nenhum com força para deslocar o ajuste sozinho; e foi só em `Credit`, com dois preditores que andam quase colados, que a colinearidade apareceu, alargando em mais de doze vezes a faixa de coeficientes que quase empatam em RSS. Cada um pede o gráfico certo para se revelar — nenhum aparece só olhando R².

## Regressão Linear contra k-Vizinhos

> **📌 Nota**
>
> Esta seção corresponde à seção 3.5 de James et al. (2023).

O capítulo inteiro apostou na mesma forma: uma combinação linear de coeficientes, seja a reta simples da seção 8.1, o plano múltiplo da 8.3, ou a curva de grau 5 da 8.5, que continua sendo linear nos coeficientes mesmo com `potencia²` dentro dela. A seção 7.3 já mostrou que essa não é a única aposta possível — o k-NN não assume forma nenhuma para `f`, e deixa a vizinhança de cada ponto decidir o valor previsto ali —, e a seção 7.6 mediu o preço dessa liberdade em MSE de teste, contra uma `f` simulada e conhecida. Esta seção junta as duas coisas: a mesma régua da 7.6, aplicada à pergunta que a 7.3 deixou em aberto — quando a forma que a reta assume compensa, e quando ela custa caro?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

plt.style.use("estilo-figuras.mplstyle")

### Quando a forma verdadeira é a da reta

Responder a pergunta com `Auto` ou `Credit` não dá: a `f` que gerou `milhas_por_galao` ou `saldo` é desconhecida, e sem ela não há como saber se um ajuste chegou perto da relação verdadeira ou só decorou o ruído da amostra — a mesma dificuldade que a seção 7.6 já contornou simulando o próprio dado. Esta seção simula de novo, com a semente `np.random.default_rng(8)` deste capítulo, começando pelo cenário mais favorável à reta que existe: uma verdade exatamente linear.

In [ ]:
def f_verdadeiro(x, curvatura, freq=0.8):
    return 2.0 + 1.5 * x + curvatura * np.sin(freq * x)

rng = np.random.default_rng(8)
n_treino = 50
ruido_padrao = 0.5

x_treino = rng.uniform(-3, 3, size=n_treino)
ruido_treino = rng.normal(0, ruido_padrao, size=n_treino)
y_treino_linear = f_verdadeiro(x_treino, 0.0) + ruido_treino

n_treino, ruido_padrao

Cinquenta pontos de treino, `x` uniforme entre -3 e 3, com ruído normal de desvio padrão 0,5 somado a uma reta verdadeira, `f(x) = 2,0 + 1,5x`. A função `f_verdadeiro` carrega um segundo termo, `curvatura * sin(0,8x)`, que fica zerado por enquanto — a próxima seção liga esse termo aos poucos, sem mudar mais nada.

In [ ]:
reta = LinearRegression().fit(x_treino.reshape(-1, 1), y_treino_linear)
knn1 = KNeighborsRegressor(n_neighbors=1).fit(x_treino.reshape(-1, 1), y_treino_linear)
knn9 = KNeighborsRegressor(n_neighbors=9).fit(x_treino.reshape(-1, 1), y_treino_linear)

mse_treino_reta = mean_squared_error(y_treino_linear, reta.predict(x_treino.reshape(-1, 1)))
mse_treino_knn1 = mean_squared_error(y_treino_linear, knn1.predict(x_treino.reshape(-1, 1)))
mse_treino_knn9 = mean_squared_error(y_treino_linear, knn9.predict(x_treino.reshape(-1, 1)))

coef_reta = float(reta.coef_[0])
intercepto_reta = float(reta.intercept_)

(
    round(mse_treino_reta, 2),
    round(mse_treino_knn1, 2),
    round(mse_treino_knn9, 2),
    round(coef_reta, 2),
    round(intercepto_reta, 2),
)

Sobre o próprio treino, a reta erra 0,26 de MSE; o k-NN com k=1 erra 0,00 — cada ponto é seu próprio vizinho mais próximo, então a previsão para ele é a resposta que ele mesmo tinha —; k=9 erra 0,31, mais que a reta. O coeficiente que a reta encontrou, 1,50, praticamente repete o 1,5 verdadeiro; o intercepto, 2,08, fica perto do 2,0 verdadeiro, com a folga vindo só do ruído desses cinquenta pontos.

In [ ]:
n_teste_grande = 20_000
x_teste_grande = rng.uniform(-3, 3, size=n_teste_grande)
ruido_teste_grande = rng.normal(0, ruido_padrao, size=n_teste_grande)
y_teste_grande_linear = f_verdadeiro(x_teste_grande, 0.0) + ruido_teste_grande

n_teste_grande

Vinte mil pontos novos, nunca usados no ajuste, só para medir o que vem a seguir com a precisão que cinquenta pontos não entregam — a mesma razão que levou a seção 7.6 a gerar cinco mil, e a 7.7 a gerar vinte mil.

In [ ]:
ks = [1, 3, 5, 7, 9, 13, 19, 27, 39]
mse_teste_reta_linear = mean_squared_error(
    y_teste_grande_linear, reta.predict(x_teste_grande.reshape(-1, 1))
)
mses_teste_knn_linear = np.array([
    mean_squared_error(
        y_teste_grande_linear,
        KNeighborsRegressor(n_neighbors=k)
        .fit(x_treino.reshape(-1, 1), y_treino_linear)
        .predict(x_teste_grande.reshape(-1, 1)),
    )
    for k in ks
])

melhor_k_linear = int(ks[int(np.argmin(mses_teste_knn_linear))])
melhor_mse_knn_linear = float(mses_teste_knn_linear.min())
reta_vence_sempre_quando_linear = bool(mse_teste_reta_linear < mses_teste_knn_linear.min())

(
    round(float(mse_teste_reta_linear), 2),
    melhor_k_linear,
    round(melhor_mse_knn_linear, 2),
    round(float(mses_teste_knn_linear[0]), 2),
    reta_vence_sempre_quando_linear,
)

In [ ]:
# Figura: Esquerda: cinquenta pontos de treino, a reta verdadeira (que aqui coincide com f, pois curvatura=0), a reta ajustada (tracejada) e os ajustes de k-NN com k=1 e k=9 — o mesmo método, só a espessura da linha muda com k. Direita: MSE de teste (vinte mil pontos) contra 1/k, em escala log; a reta ajustada é a linha tracejada horizontal, porque ela não depende de k. Em nenhum dos nove valores de k o k-NN desce abaixo dela.
grade = np.linspace(x_teste_grande.min(), x_teste_grande.max(), 400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))

ax1.scatter(x_treino, y_treino_linear, color="C0", s=18, alpha=0.6, label="treino")
ax1.plot(grade, f_verdadeiro(grade, 0.0), color="C2", linewidth=2.4, label="f verdadeiro")
ax1.plot(
    grade, reta.predict(grade.reshape(-1, 1)),
    color="C1", linewidth=2, linestyle="--", label="reta",
)
ax1.plot(grade, knn9.predict(grade.reshape(-1, 1)), color="C3", linewidth=2.4, label="k-NN, k=9")
ax1.plot(grade, knn1.predict(grade.reshape(-1, 1)), color="C3", linewidth=0.9, label="k-NN, k=1")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.legend(fontsize=7, loc="upper left")

inverso_k = 1 / np.array(ks)
ax2.plot(inverso_k, mses_teste_knn_linear, color="C3", linewidth=2, marker="o", markersize=4, label="k-NN")
ax2.axhline(mse_teste_reta_linear, color="C1", linewidth=2, linestyle="--", label="reta")
ax2.set_xscale("log")
ax2.set_xlabel("1/k (escala log)")
ax2.set_ylabel("MSE de teste")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

No teste grande, a reta erra 0,25 de MSE; o menor erro do k-NN, entre os nove valores de k varridos, sai em k=7 (`melhor_k_linear`) e chega a 0,35 — acima do 0,25 da reta. `reta_vence_sempre_quando_linear` confirma `True`: nenhum dos nove valores de k derruba a reta quando a verdade é dela mesma. O contraste mais direto é com k=1: 0,00 de erro no treino, 0,47 no teste — o mesmo ajuste que decorou a amostra de cinquenta pontos é o que mais erra em dado novo, a mesma lição de overfitting que a seção 7.3 já tinha nomeado.

### Quando a curvatura cresce

A verdade linear é o caso mais favorável à reta que existe — nenhuma forma bate a forma certa. Dado real raramente entrega isso. A mesma `f_verdadeiro` usada acima carrega o termo `curvatura * sin(0,8x)`, deixado em zero até aqui; ligando-o aos poucos, sem tocar nos cinquenta pontos de treino nem no ruído que os acompanha, a única coisa que muda entre uma rodada e a próxima é o quanto a verdade se afasta de uma reta.

In [ ]:
curvaturas = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0])
mses_reta_curvatura = []
mses_knn_curvatura = []
for c in curvaturas:
    y_treino_c = f_verdadeiro(x_treino, c) + ruido_treino
    reta_c = LinearRegression().fit(x_treino.reshape(-1, 1), y_treino_c)
    ruido_teste_c = rng.normal(0, ruido_padrao, size=n_teste_grande)
    y_teste_c = f_verdadeiro(x_teste_grande, c) + ruido_teste_c

    mse_reta_c = mean_squared_error(y_teste_c, reta_c.predict(x_teste_grande.reshape(-1, 1)))
    mses_knn_c = [
        mean_squared_error(
            y_teste_c,
            KNeighborsRegressor(n_neighbors=k)
            .fit(x_treino.reshape(-1, 1), y_treino_c)
            .predict(x_teste_grande.reshape(-1, 1)),
        )
        for k in ks
    ]
    mses_reta_curvatura.append(mse_reta_c)
    mses_knn_curvatura.append(min(mses_knn_c))

mses_reta_curvatura = np.array(mses_reta_curvatura)
mses_knn_curvatura = np.array(mses_knn_curvatura)
knn_vence = mses_knn_curvatura < mses_reta_curvatura

indice_transicao = int(np.argmax(knn_vence))
curvatura_antes = float(curvaturas[indice_transicao - 1])
curvatura_depois = float(curvaturas[indice_transicao])
transicao_e_unica = bool(np.all(knn_vence[indice_transicao:]))

(
    round(float(mses_reta_curvatura[0]), 2),
    round(float(mses_reta_curvatura[-1]), 2),
    round(float(mses_knn_curvatura[0]), 2),
    round(float(mses_knn_curvatura[-1]), 2),
    curvatura_antes,
    round(float(mses_reta_curvatura[indice_transicao - 1]), 2),
    round(float(mses_knn_curvatura[indice_transicao - 1]), 2),
    curvatura_depois,
    round(float(mses_reta_curvatura[indice_transicao]), 2),
    round(float(mses_knn_curvatura[indice_transicao]), 2),
    transicao_e_unica,
)

In [ ]:
# Figura: MSE de teste da reta e do melhor k-NN de cada nível (o menor entre os nove k da seção anterior), contra a curvatura da verdade simulada. A reta piora sem parar; o k-NN quase não se mexe. A faixa sombreada marca onde a curva da reta cruza a do k-NN, entre as curvaturas testadas.
fig, ax = plt.subplots()
ax.axvspan(curvatura_antes, curvatura_depois, color="C2", alpha=0.15)
ax.plot(curvaturas, mses_reta_curvatura, color="C1", linewidth=2, marker="o", markersize=4, label="reta")
ax.plot(curvaturas, mses_knn_curvatura, color="C3", linewidth=2, marker="o", markersize=4, label="k-NN (melhor k)")
ax.set_xlabel("curvatura")
ax.set_ylabel("MSE de teste")
ax.legend()
plt.tight_layout()
plt.show()

Nos onze níveis varridos, a reta piora sem parar — de 0,26 em curvatura 0,0 a 0,84 em curvatura 3,0 —, enquanto o melhor k-NN de cada nível mal se mexe: 0,35 no começo, 0,36 no fim. A curva da reta cruza a do k-NN entre curvatura 1,0, onde a reta ainda vence (0,31 contra 0,34), e 1,3, onde o k-NN passa à frente (0,35 contra 0,34). `transicao_e_unica` confirma `True`: em nenhum nível mais curvo depois desse o k-NN perde a dianteira de volta — não é um cruzamento de ida e volta que a sorte da amostra desfaz. Com um conjunto de teste de poucas centenas de pontos, essa margem — 0,01 entre as duas curvas bem no cruzamento — teria variância grande o bastante para inventar ou esconder uma travessia; vinte mil pontos bastam para o cruzamento não depender do sorteio.

### A maldição da dimensionalidade

A curvatura mais alta da varredura acima, 2,5 — no trecho onde o k-NN já vencia com folga —, serve de base para a última pergunta: o que acontece quando se soma preditores que não têm relação nenhuma com a resposta?

In [ ]:
curvatura_alta = 2.5
y_treino_dim = f_verdadeiro(x_treino, curvatura_alta) + ruido_treino

ps = [1, 2, 3, 4, 6, 10, 15, 20]
p_max = max(ps)
ruido_extra_treino = rng.uniform(-3, 3, size=(n_treino, p_max - 1))
ruido_extra_teste = rng.uniform(-3, 3, size=(n_teste_grande, p_max - 1))
ruido_teste_dim = rng.normal(0, ruido_padrao, size=n_teste_grande)
y_teste_dim = f_verdadeiro(x_teste_grande, curvatura_alta) + ruido_teste_dim

mses_reta_p = []
mses_knn_p = []
for p in ps:
    if p == 1:
        X_treino_p = x_treino.reshape(-1, 1)
        X_teste_p = x_teste_grande.reshape(-1, 1)
    else:
        X_treino_p = np.column_stack([x_treino, ruido_extra_treino[:, : p - 1]])
        X_teste_p = np.column_stack([x_teste_grande, ruido_extra_teste[:, : p - 1]])

    reta_p = LinearRegression().fit(X_treino_p, y_treino_dim)
    mse_reta_p = mean_squared_error(y_teste_dim, reta_p.predict(X_teste_p))
    mses_knn_p_k = [
        mean_squared_error(
            y_teste_dim,
            KNeighborsRegressor(n_neighbors=k).fit(X_treino_p, y_treino_dim).predict(X_teste_p),
        )
        for k in ks
    ]
    mses_reta_p.append(mse_reta_p)
    mses_knn_p.append(min(mses_knn_p_k))

mses_reta_p = np.array(mses_reta_p)
mses_knn_p = np.array(mses_knn_p)

razao_knn = float(mses_knn_p[-1] / mses_knn_p[0])
razao_reta = float(mses_reta_p[-1] / mses_reta_p[0])
knn_degrada_muito_mais = bool(razao_knn > razao_reta)
knn_vence_em_p1 = bool(mses_knn_p[0] < mses_reta_p[0])
reta_vence_do_p2_em_diante = bool(np.all(mses_reta_p[1:] < mses_knn_p[1:]))

(
    round(float(mses_reta_p[0]), 2),
    round(float(mses_knn_p[0]), 2),
    round(float(mses_reta_p[-1]), 2),
    round(float(mses_knn_p[-1]), 2),
    round(razao_reta, 2),
    round(razao_knn, 2),
    knn_vence_em_p1,
    reta_vence_do_p2_em_diante,
    knn_degrada_muito_mais,
)

In [ ]:
# Figura: MSE de teste da reta e do melhor k-NN contra o número de preditores p, com a mesma verdade fortemente não linear em x1 e p-1 preditores extras sem relação nenhuma com a resposta. Eixo y em escala log, porque as duas curvas crescem em ritmos muito diferentes. A faixa sombreada marca a passagem de p=1 para p=2, onde a vantagem do k-NN desaparece.
fig, ax = plt.subplots()
ax.axvspan(1, 2, color="C2", alpha=0.15)
ax.plot(ps, mses_reta_p, color="C1", linewidth=2, marker="o", markersize=4, label="reta")
ax.plot(ps, mses_knn_p, color="C3", linewidth=2, marker="o", markersize=4, label="k-NN (melhor k)")
ax.set_yscale("log")
ax.set_xlabel("número de preditores (p)")
ax.set_ylabel("MSE de teste (escala log)")
ax.legend()
plt.tight_layout()
plt.show()

Com um preditor só, `knn_vence_em_p1` confirma `True`: o k-NN erra 0,35 contra 0,66 da reta. Basta somar um preditor de ruído para a ordem se inverter, e `reta_vence_do_p2_em_diante` confirma que ela segue à frente em todos os sete valores de p maiores testados. Nos vinte preditores — dezenove deles pura decoração —, a reta chega a 1,07, 1,63 vez o erro que tinha com um preditor só; o k-NN chega a 13,37, 38,64 vezes o que tinha. `knn_degrada_muito_mais` confirma `True`. Nenhum dos dezenove preditores extras carrega informação sobre a resposta, mas a distância que o k-NN usa para achar "vizinho mais próximo" soma a contribuição de toda coordenada, relevante ou não; em vinte dimensões, os cinquenta pontos de treino que bastavam para cobrir bem uma única reta ficam espalhados demais para que algum fique de fato perto de um ponto novo — a vizinhança deixa de significar proximidade. A reta não paga esse preço: um coeficiente perto de zero em cada preditor inútil já resolve o problema, sem comprometer o que interessa.

### Nenhum dos dois vence sozinho

Três medições, três respostas diferentes: a reta venceu quando a verdade era dela mesma; o k-NN passou à frente a partir de certa curvatura, contanto que a dimensão ficasse baixa; e um único preditor extra sem relação com a resposta já bastou para devolver a vantagem à reta. Qual dos dois ajustar a um dado novo é uma pergunta que só se responde conhecendo a forma verdadeira de `f` — e essa é exatamente a informação que falta em qualquer problema real. Escolher sem essa certeza é o que a validação cruzada, no capítulo 10, ensina a fazer.

## Leituras adicionais

*A escrever.*

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.